# Track A — S24 SD LoRA VS Generation (Colab)

**전제조건**:
- 런타임 유형: GPU (T4 이상 권장)
- S24 EEG 데이터는 GitHub 저장소에 `.npz` 형식으로 포함되어 있음 (Drive 불필요)
- S24 SupCon 체크포인트 (`subj24_best.pt`)도 저장소에 포함

**학습 결과 체크포인트는 Google Drive에 저장됩니다** (세션 종료 후 유지):
- Drive 경로: `MyDrive/vsvi_data/checkpoints_vsre_lora_gen/`

**순서**: 셀을 위에서 아래로 순서대로 실행

In [ ]:
# [1] GPU 확인
import torch
print(f'torch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경 > GPU 선택')

In [ ]:
# [2] 코드 clone (S24 .npz 데이터 포함)
import os, subprocess

REPO    = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

if not os.path.exists(WORKDIR):
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)
else:
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'], check=True)

os.chdir(WORKDIR)
print(f'Working dir: {os.getcwd()}')
subprocess.run(['git', 'log', '--oneline', '-3'])

# S24 .npz 확인
npz_files = sorted([f for f in os.listdir('preproc_vs_re') if 'subj_24' in f and f.endswith('.npz')])
mat_files = sorted([f for f in os.listdir('preproc_vs_re') if 'subj_24' in f and f.endswith('.mat')])
print(f'S24 .npz files: {len(npz_files)}')
print(f'S24 .mat files: {len(mat_files)}')
for f in npz_files:
    print(f'  {f}')

if len(npz_files) == 0 and len(mat_files) == 0:
    raise FileNotFoundError('S24 데이터 없음. GitHub push 상태 확인 필요')

In [ ]:
# [3] 패키지 설치
# torchao 먼저 업그레이드: peft>=0.15 가 torchao>=0.16 을 요구함
subprocess.run(['pip', 'install', '-q', 'torchao', '--upgrade'], check=True)
subprocess.run(['pip', 'install', '-q',
                'peft', 'diffusers', 'accelerate', 'transformers', 'h5py'], check=True)

import peft, diffusers
print(f'peft {peft.__version__}, diffusers {diffusers.__version__}')

In [ ]:
# [4] Google Drive 마운트 (체크포인트 저장용)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/vsvi_data'
os.makedirs(DRIVE_DATA, exist_ok=True)

# 체크포인트 출력 디렉토리 (Drive에 저장, 세션 종료 후에도 유지)
ckpt_out = f'{DRIVE_DATA}/checkpoints_vsre_lora_gen'
os.makedirs(ckpt_out, exist_ok=True)

dst_ckpt = f'{WORKDIR}/checkpoints_vsre_lora_gen'
if not os.path.exists(dst_ckpt):
    os.symlink(ckpt_out, dst_ckpt)
print(f'checkpoints_vsre_lora_gen -> {ckpt_out}')

In [ ]:
# [5] Preflight
os.chdir(WORKDIR)
ret = os.system(
    'python preflight_track_a.py '
    '--subject_id 24 '
    '--img_root preproc_data_vi/images '
    '--supcon_ckpt checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon '
    '--data_root preproc_vs_re '
    '--ckpt_root checkpoints_vsre_lora_gen '
    '--check_data'
)
assert ret == 0, 'Preflight FAILED -- 위 출력 확인 후 문제 해결'

In [ ]:
# [6] S24 LoRA r=16 학습
# 예상 소요시간: T4 기준 약 8~12시간
import subprocess

os.chdir(WORKDIR)
cmd = [
    'python', 'train_vs_re_lora_gen.py',
    '--subject_ids', '24',
    '--lora_r', '16',
    '--n_eeg_tokens', '16',
    '--epochs', '100',
    '--img_root', 'preproc_data_vi/images',
    '--supcon_ckpt', 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon',
    '--ckpt_root', 'checkpoints_vsre_lora_gen',
]
print('Command:', ' '.join(cmd))
print('Starting...')
result = subprocess.run(cmd)
print(f'\nExit code: {result.returncode}')

In [ ]:
# [7] S24 LoRA r=32 학습 (r=16 완료 후 실행)
os.chdir(WORKDIR)
cmd = [
    'python', 'train_vs_re_lora_gen.py',
    '--subject_ids', '24',
    '--lora_r', '32',
    '--n_eeg_tokens', '16',
    '--epochs', '100',
    '--img_root', 'preproc_data_vi/images',
    '--supcon_ckpt', 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon',
    '--ckpt_root', 'checkpoints_vsre_lora_gen',
]
print('Command:', ' '.join(cmd))
result = subprocess.run(cmd)
print(f'\nExit code: {result.returncode}')

In [ ]:
# [8] 결과 확인
import glob
ckpt_out = '/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen'
results = glob.glob(f'{ckpt_out}/**/results_dual_gallery.csv', recursive=True)
for r in sorted(results):
    print(r)
    with open(r) as f:
        print(f.read())